# Initial data exploration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Support running from either the project root or the notebook directory.
project_root = Path.cwd()
if not (project_root / "data" / "wsh_pitches.csv").exists():
    project_root = project_root.parent

pitches = pd.read_csv(project_root / "data" / "wsh_pitches.csv")
pitches.head()

### Data missingness checks and confirming binary columns

In [ ]:
total_pitches = len(pitches)
print(f"Total pitches: {total_pitches:,}")

rate_columns = ["is_fb", "is_in_zone", "is_swing", "is_miss"]
display(pitches.isna().sum().rename("missing_count").to_frame())
display(pitches[rate_columns].agg(["min", "max"]))

assert not pitches.isna().any().any(), "Review missing values before computing fractions."
assert pitches[rate_columns].isin([0, 1]).all().all()
assert pitches[["throw_side", "bat_side"]].isin(["L", "R"]).all().all()
assert (pitches["is_miss"] <= pitches["is_swing"]).all()

### Splits by bat/throw side

In [ ]:
throw_counts = pitches["throw_side"].value_counts().sort_index()
throw_summary = pd.DataFrame({
    "pitch_count": throw_counts,
    "fraction": throw_counts / total_pitches,
})
throw_summary

In [ ]:
bat_counts = pitches["bat_side"].value_counts().sort_index()
bat_summary = pd.DataFrame({
    "pitch_count": bat_counts,
    "fraction": bat_counts / total_pitches,
})
bat_summary

In [ ]:
handedness_counts = pitches.groupby(["throw_side", "bat_side"]).size()
handedness_summary = pd.DataFrame({
    "pitch_count": handedness_counts,
    "fraction": handedness_counts / total_pitches,
})
handedness_summary

### Pitch fractions, overall and by platoon

`contact_rate` = (swings − misses) / swings, computed within each group. It is undefined (NaN) for groups with no swings. The other fractions use all pitches as the denominator.

In [ ]:
rate_names = {
    "is_fb": "fastball_fraction",
    "is_in_zone": "in_zone_fraction",
    "is_swing": "swing_fraction",
    "is_miss": "swing_and_miss_fraction",
}
overall_rates = pitches[rate_columns].mean().rename(index=rate_names)
swing_count = pitches["is_swing"].sum()
miss_count = pitches["is_miss"].sum()
overall_rates["contact_rate"] = (
    (swing_count - miss_count) / swing_count if swing_count > 0 else float("nan")
)
overall_rates.to_frame("fraction")

In [ ]:
rates_by_throw = pitches.groupby("throw_side")[rate_columns].mean().rename(columns=rate_names)
rates_by_throw.insert(0, "pitch_count", throw_counts)

swing_miss_counts = pitches.groupby("throw_side")[["is_swing", "is_miss"]].sum()
rates_by_throw["contact_rate"] = (
    (swing_miss_counts["is_swing"] - swing_miss_counts["is_miss"])
    / swing_miss_counts["is_swing"].where(swing_miss_counts["is_swing"] > 0)
)
rates_by_throw

In [ ]:
rates_by_bat = pitches.groupby("bat_side")[rate_columns].mean().rename(columns=rate_names)
rates_by_bat.insert(0, "pitch_count", bat_counts)

swing_miss_counts = pitches.groupby("bat_side")[["is_swing", "is_miss"]].sum()
rates_by_bat["contact_rate"] = (
    (swing_miss_counts["is_swing"] - swing_miss_counts["is_miss"])
    / swing_miss_counts["is_swing"].where(swing_miss_counts["is_swing"] > 0)
)
rates_by_bat

In [ ]:
rates_by_matchup = (
    pitches.groupby(["throw_side", "bat_side"])[rate_columns]
    .mean()
    .rename(columns=rate_names)
)
rates_by_matchup.insert(0, "pitch_count", handedness_counts)

swing_miss_counts = pitches.groupby(["throw_side", "bat_side"])[["is_swing", "is_miss"]].sum()
rates_by_matchup["contact_rate"] = (
    (swing_miss_counts["is_swing"] - swing_miss_counts["is_miss"])
    / swing_miss_counts["is_swing"].where(swing_miss_counts["is_swing"] > 0)
)
rates_by_matchup

### Pitches per pitcher, batter, and pitcher–batter pair

In [ ]:
pitcher_counts = pitches.groupby("pitcher_id").size().rename("pitch_count")
batter_counts = pitches.groupby("batter_id").size().rename("pitch_count")
pair_counts = pitches.groupby(["pitcher_id", "batter_id"]).size().rename("pitch_count")

print(f"Unique pitchers: {len(pitcher_counts):,}")
print(f"Unique batters: {len(batter_counts):,}")
print(f"Unique pitcher–batter pairs: {len(pair_counts):,}")

pd.DataFrame({
    "pitcher": pitcher_counts.describe(),
    "batter": batter_counts.describe(),
    "pitcher_batter_pair": pair_counts.describe(),
})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, counts, label in zip(
    axes,
    [pitcher_counts, batter_counts, pair_counts],
    ["pitcher", "batter", "pitcher–batter pair"],
):
    ax.hist(counts, bins=30, edgecolor="white")
    ax.set_title(f"Pitches per {label}")
    ax.set_xlabel("Number of pitches")
    ax.set_ylabel(f"Number of {label}s")

fig.tight_layout()
plt.show()

So far, all the data looks reasonable and as expected, with one exception: contact rates appear to be more extreme than expected based on platoon matchups (in both directions). Same-side matchups have considerably lower contact rates than I'd expect (despite the fact that they're same-side matchups and therefore favor the pitcher), and opposite-side matchups have considerably higher contract rates than I'd expect (also despite the fact that they're opposite-side matchups and therefore favor the batter). Swing rates also look more extreme by platoon than I'd expect, though to a lesser degree. Since this is the data provided for this project, I'll treat it as ground truth, but this is something I'll want to keep flagged mentally as I evaluate the model I eventually build within the context of baseball models I've typically seen/built before.

Before proceeding with any actual modeling choices, I want to do one further investigation for the platoon contact discrepancy. Is there something more specific going on with IZ vs OOZ swing/contact rates, or does my current observation of the contact rate that isn't conditioned on IZ/OOZ hold for both?

In [ ]:
# Compare swing/contact rates within each zone and platoon matchup.
# Swing fraction uses pitches; contact rate uses swings as its denominator.
zone_matchup_groups = pitches.groupby(["is_in_zone", "throw_side", "bat_side"])
zone_matchup_rates = zone_matchup_groups.agg(
    pitch_count=("is_swing", "size"),
    swing_count=("is_swing", "sum"),
    miss_count=("is_miss", "sum"),
    swing_fraction=("is_swing", "mean"),
)
zone_matchup_rates["contact_rate"] = (
    (zone_matchup_rates["swing_count"] - zone_matchup_rates["miss_count"])
    / zone_matchup_rates["swing_count"].where(zone_matchup_rates["swing_count"] > 0)
)
zone_matchup_rates = zone_matchup_rates.rename(
    index={0: "OOZ", 1: "IZ"}, level="is_in_zone"
).rename_axis(index={"is_in_zone": "zone"})
zone_matchup_rates


Looks like this observation holds true across both IZ and OOZ. I'll proceed with my modeling as if this were ground truth for now, but I'll keep my eye out for what (potential) strange implications this has on my model downstream.

## Contact modeling: target, assumptions, and configuration

We treat the supplied data, including its unusual platoon contact patterns, as ground truth. The target is **P(contact | swing, batter ID, pitcher ID, batting side, throwing side, in-zone status, fastball status)**. Takes are excluded from the likelihood. All four context features (bat/throw side, IZ, and FB) and their interactions enter immediately through 16 population context logits. Shared batter and pitcher intercepts pool across contexts; there are no player-specific context slopes.

For swing *i*, contact is Bernoulli with logit probability `context_logit + batter_effect + pitcher_effect`. Context logits have independent N(0, 2.5²) priors (this would be treated as a hyperparameter to tune in a more robust version of this analysis). Player effects have independent N(0, 1 / precision) priors; batter and pitcher precisions are selected on validation log loss. The likelihood assumes conditional independence and stable player ability. This is a Gaussian-shrinkage MAP fit with a conditional Laplace approximation, not MCMC.

The proper priors resolve the additive likelihood's non-identifiability. For interpretation, player effects are subsequently centered within their observed handedness, and the group means absorbed into context logits. **Neutral opposition** means zero centered deviation: the equal-player mean logit effect within the requested side, rather than the particular opponents a hitter faced. Neutral-player uncertainty includes the uncertainty in that group mean, not the variability of a randomly selected actual opponent.

Keep the existing `.venv` and dependencies. Run this notebook from the project root or `notebook/`. Seeds, split assignments, selected settings, scores, and fitted coefficients are saved under ignored `outputs/contact_model/`.

In [ ]:
import hashlib
import json
import sys
from itertools import product
from time import perf_counter

import numpy as np
from scipy.special import expit
from threadpoolctl import threadpool_limits

# Bound BLAS threads for these small matrix operations; no GPU is needed.
blas_limit = threadpool_limits(limits=1, user_api="blas")
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))
from contact_model import (
    CONTEXT,
    CONTEXT_LEVELS,
    ContactModel,
    batter_projection,
    context_projection,
    pair_sensitivity_split,
    performance,
    predictive_context_check,
    prepare_swings,
    reference_contexts,
    reliability,
    select_shrinkage,
    simulation_recovery,
    split_swings,
    subgroup_performance,
)

SPLIT_SEED = 42
DRAW_SEED = 44
PRECISION_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]
CONTEXT_SD = 2.5
VALIDATION_TOLERANCE = 0.001
POSTERIOR_DRAWS = 2000
SIMULATION_REPETITIONS = 20
output_dir = project_root / "outputs" / "contact_model"
output_dir.mkdir(parents=True, exist_ok=True)
swings = prepare_swings(pitches)
print(f"Modeling {len(swings):,} swings from {len(pitches):,} pitches.")

### Train / validation / test: individual coverage

Use a 60/20/20 swing-level split stratified by the 16 contexts. Require every batter and pitcher in every partition; seed retries depend only on IDs and context, never contact outcomes. Player context-specific coverage is not required. Player encodings and model fitting use training data only.

The primary target is additional swings involving known players. Pairs may cross partitions, so residual matchup dependence may make this evaluation optimistic; a separate pair-grouped sensitivity check uses development data only. No dates or game identifiers are available, so neither design establishes chronological or game-independent forecast performance.

In [ ]:
train, validation, test, used_split_seed = split_swings(swings, SPLIT_SEED)
development = pd.concat([train, validation]).sort_index()
assert set(train.index).isdisjoint(validation.index)
assert set(development.index).isdisjoint(test.index)
assert set(development.index) | set(test.index) == set(swings.index)

split_summary = pd.DataFrame({
    name: {
        "swings": len(frame),
        "batters": frame.batter_id.nunique(),
        "pitchers": frame.pitcher_id.nunique(),
        "min_batter_swings": frame.groupby("batter_id").size().min(),
        "min_pitcher_swings": frame.groupby("pitcher_id").size().min(),
    }
    for name, frame in [("train", train), ("validation", validation), ("test", test)]
}).T
display(split_summary)
display(pd.concat({
    name: frame.groupby(CONTEXT).size()
    for name, frame in [("train", train), ("validation", validation), ("test", test)]
}, axis=1))

split_assignment = pd.concat([
    pd.Series(name, index=frame.index)
    for name, frame in [("train", train), ("validation", validation), ("test", test)]
]).sort_index().rename("partition")
split_assignment.rename_axis("original_pitch_row").to_csv(output_dir / "split_assignments.csv")
print(f"Frozen split seed: {used_split_seed}. No test outcomes are used in selection.")

### Validation-selected shrinkage and meaningful baselines

Compare an overall contact-rate baseline, a context-only model, a context-plus-batter model, and the context-plus-batter-plus-pitcher model. Tune batter-only precision separately and use a small Cartesian grid for the full model. Scores use unweighted swings; also report batter-balanced scores. The agreed full architecture is retained, with its precisions selected using validation log loss. Among settings within 0.001 of the minimum, prefer the strongest overall shrinkage (smallest summed player-effect prior variance). This protects against weakly identified scales when negligible score differences favor nearly unregularized estimates. Test outcomes are not inspected here.

The summed likelihood makes precision an absolute prior precision: standard deviation is `1 / sqrt(precision)`. This is not a regularization strength that is rescaled by sample size. Very similar validation scores should not be interpreted as precise identification of the shrinkage scale.

The candidate table displays `summed_prior_variance = 1 / batter_precision + 1 / pitcher_precision`, the log-loss gap from the best score, tolerance eligibility, and the selected row. A smaller summed variance means stronger overall prior shrinkage; it is a selection-strength metric, not an observed percentage of shrinkage for any individual player.

In [ ]:
selection_start = perf_counter()
context_training_model = ContactModel(
    include_batter=False, include_pitcher=False, context_sd=CONTEXT_SD
).fit(train)
validation_baselines = {
    "Overall": performance(validation, np.full(len(validation), train.contact.mean())),
    "Context": performance(validation, context_training_model.predict(validation)),
}

batter_trials = []
for precision in PRECISION_GRID:
    fitted = ContactModel(
        batter_precision=precision, include_pitcher=False, context_sd=CONTEXT_SD
    ).fit(train)
    batter_trials.append({
        "batter_precision": precision,
        **performance(validation, fitted.predict(validation)),
    })
batter_selection = pd.DataFrame(batter_trials).sort_values("log_loss")
chosen_batter_only = select_shrinkage(batter_selection, VALIDATION_TOLERANCE)
chosen_batter_only_precision = float(chosen_batter_only.batter_precision)
validation_baselines["Context + batter"] = chosen_batter_only.drop("batter_precision").to_dict()

full_trials = []
for batter_precision, pitcher_precision in product(PRECISION_GRID, repeat=2):
    fitted = ContactModel(batter_precision, pitcher_precision, context_sd=CONTEXT_SD).fit(train)
    full_trials.append({
        "batter_precision": batter_precision,
        "pitcher_precision": pitcher_precision,
        "fit_seconds": fitted.fit_seconds,
        "max_gradient": fitted.gradient_max,
        **performance(validation, fitted.predict(validation)),
    })
shrinkage_selection = pd.DataFrame(full_trials).sort_values("log_loss")
chosen = select_shrinkage(shrinkage_selection, VALIDATION_TOLERANCE)
chosen_batter_precision = float(chosen.batter_precision)
chosen_pitcher_precision = float(chosen.pitcher_precision)
validation_baselines["Context + batter + pitcher"] = chosen.drop(
    ["batter_precision", "pitcher_precision", "fit_seconds", "max_gradient"]
).to_dict()
validation_scores = pd.DataFrame(validation_baselines).T

shrinkage_selection["summed_prior_variance"] = (
    1 / shrinkage_selection.batter_precision + 1 / shrinkage_selection.pitcher_precision
)
shrinkage_selection["log_loss_gap"] = shrinkage_selection.log_loss - shrinkage_selection.log_loss.min()
shrinkage_selection["within_tolerance"] = (
    shrinkage_selection.log_loss <= shrinkage_selection.log_loss.min() + VALIDATION_TOLERANCE
)
shrinkage_selection["selected"] = (
    shrinkage_selection.batter_precision.eq(chosen_batter_precision)
    & shrinkage_selection.pitcher_precision.eq(chosen_pitcher_precision)
)
display(validation_scores)
display(shrinkage_selection[[
    "batter_precision", "pitcher_precision", "log_loss", "log_loss_gap",
    "summed_prior_variance", "within_tolerance", "selected",
]].style.format({
    "log_loss": "{:.6f}", "log_loss_gap": "{:.6f}", "summed_prior_variance": "{:.2f}",
}).hide(axis="index"))
print(
    f"Chosen log-loss gap: {chosen.log_loss - shrinkage_selection.log_loss.min():.6f} "
    f"(tolerance {VALIDATION_TOLERANCE:g}); summed prior variance: "
    f"{1 / chosen_batter_precision + 1 / chosen_pitcher_precision:.2f}. "
    "This is the smallest summed variance among eligible candidates."
)
print(f"Selected precisions: batter={chosen_batter_precision:g}, pitcher={chosen_pitcher_precision:g}")
print(f"Selection runtime: {perf_counter() - selection_start:.1f} seconds")
if any(value in [min(PRECISION_GRID), max(PRECISION_GRID)] for value in [chosen_batter_precision, chosen_pitcher_precision]):
    print("A chosen precision is at a grid boundary; inspect score sensitivity before treating it as well determined.")
shrinkage_selection.to_csv(output_dir / "shrinkage_validation.csv", index=False)
validation_scores.to_csv(output_dir / "validation_scores.csv")

fig, ax = plt.subplots(figsize=(6, 4))
grid = shrinkage_selection.pivot(index="batter_precision", columns="pitcher_precision", values="log_loss")
image = ax.imshow(grid.to_numpy(), aspect="auto", origin="lower", cmap="coolwarm")
ax.scatter(grid.columns.get_loc(chosen_pitcher_precision), grid.index.get_loc(chosen_batter_precision), marker="*", s=140, color="black", label="Selected")
ax.set_xticks(range(len(grid.columns)), grid.columns)
ax.set_yticks(range(len(grid.index)), grid.index)
ax.set_xlabel("Pitcher prior precision")
ax.set_ylabel("Batter prior precision")
ax.set_title("Validation log loss (lower is better)")
ax.legend(loc="upper left")
fig.colorbar(image, ax=ax)
fig.tight_layout()
plt.show()

### Pair-grouped sensitivity check on development data

This check changes the evaluation target to held-out pitcher–batter combinations. It uses no primary test rows and does not select the primary penalties. Its scores are descriptive: selected penalties were informed by outcomes in the development set, so this is not an additional untouched evaluation. Check individual coverage; if absent players occur, report them instead of silently substituting zero-uncertainty estimates.

In [ ]:
pair_train, pair_validation = pair_sensitivity_split(development, seed=43)
train_pairs = set(map(tuple, pair_train[["batter_id", "pitcher_id"]].to_numpy()))
validation_pairs = set(map(tuple, pair_validation[["batter_id", "pitcher_id"]].to_numpy()))
assert train_pairs.isdisjoint(validation_pairs)
missing_pair_players = {
    key: set(pair_validation[key]) - set(pair_train[key])
    for key in ["batter_id", "pitcher_id"]
}
print("Players absent from sensitivity training:", missing_pair_players)
if any(missing_pair_players.values()):
    print("Sensitivity scoring skipped: evaluating unseen players needs an explicit population-predictive treatment.")
    pair_scores = pd.DataFrame()
else:
    pair_context = ContactModel(include_batter=False, include_pitcher=False, context_sd=CONTEXT_SD).fit(pair_train)
    pair_model = ContactModel(chosen_batter_precision, chosen_pitcher_precision, context_sd=CONTEXT_SD).fit(pair_train)
    pair_scores = pd.DataFrame({
        "Context": performance(pair_validation, pair_context.predict(pair_validation)),
        "Context + batter + pitcher": performance(pair_validation, pair_model.predict(pair_validation)),
    }).T
    display(pair_scores)
    pair_scores.to_csv(output_dir / "pair_sensitivity_scores.csv")

### Refit the frozen specification and approximate uncertainty

Refit on train + validation. The test set remains excluded from all fitting and the reference mixtures conditioned on batting side. Keep this development fit for both evaluation and the batter lookup; there is no silent all-data refit.

For coefficients θ, a quadratic expansion of the log posterior around the MAP estimate gives `θ | data ≈ N(θ_MAP, H⁻¹)`, where H is the negative-log-posterior Hessian. Draw **jointly**, preserving population/player covariance. Intervals are approximate credible intervals conditional on validation-selected precision, the architecture, and the fixed reference weights. They omit uncertainty in selection and future pitch mix, and do not account for model misspecification. They describe a probability, not the realized outcome of an individual swing.

For each batting side, estimate the joint pitcher-handedness × FB distribution among that side's in-zone development swings. Weights sum to one within batting side. All hitters of the same side use the same mix, while LHH and RHH can face different typical opposition. Restrict and renormalize those weights for the LHP-only answer. We do not model an in-zone frequency in this mix: all three requested outputs condition on being in zone. The display also reports the resulting LHP share for each batting side.

In [ ]:
final_models = {
    "Context": ContactModel(include_batter=False, include_pitcher=False, context_sd=CONTEXT_SD).fit(development),
    "Context + batter": ContactModel(chosen_batter_only_precision, include_pitcher=False, context_sd=CONTEXT_SD).fit(development),
    "Context + batter + pitcher": ContactModel(chosen_batter_precision, chosen_pitcher_precision, context_sd=CONTEXT_SD).fit(development),
}
contact_model = final_models["Context + batter + pitcher"]
contact_model.laplace_covariance(development)
coefficient_draws = contact_model.draw_coefficients(POSTERIOR_DRAWS, DRAW_SEED)
reference_mix = reference_contexts(development)
display(reference_mix)
assert np.allclose(reference_mix.groupby("bat_side").weight.sum(), 1)
lhp_reference_share = reference_mix.loc[reference_mix.throw_side.eq("L")].groupby("bat_side").weight.sum()
display(lhp_reference_share.rename("LHP_share_of_IZ_swings").to_frame())
print(f"Parameters: {contact_model.n_parameters}; iterations: {contact_model.iterations}")
print(f"Max absolute gradient: {contact_model.gradient_max:.3g}")
print(f"Hessian condition number: {contact_model.hessian_condition:.1f}")

import scipy
import sklearn

run_settings = {
    "target": "contact conditional on swing", "split_seed": used_split_seed,
    "split": "60/20/20 swings stratified by context with individual coverage",
    "precision_grid": PRECISION_GRID, "validation_tolerance": VALIDATION_TOLERANCE, "batter_precision": chosen_batter_precision,
    "pitcher_precision": chosen_pitcher_precision,
    "batter_only_precision": chosen_batter_only_precision,
    "context_prior_sd": CONTEXT_SD, "draw_seed": DRAW_SEED,
    "posterior_draws": POSTERIOR_DRAWS,
    "simulation_repetitions": SIMULATION_REPETITIONS,
    "data_sha256": hashlib.sha256((project_root / "data" / "wsh_pitches.csv").read_bytes()).hexdigest(),
    "python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
    "scipy": scipy.__version__, "scikit_learn": sklearn.__version__,
    "fit_rows": len(development), "test_rows": len(test),
    "neutral_opposition": "equal-player mean logit effect within handedness",
    "reference_mix": "throw_side x is_fb conditional on bat_side and in-zone swings; development only",
    "limitations": ["no chronological validation", "conditional Laplace intervals", "historical swing mix shared within batting side"],
}
(output_dir / "run_settings.json").write_text(json.dumps(run_settings, indent=2) + "\n")
reference_mix.to_csv(output_dir / "reference_mix.csv", index=False)
np.savez_compressed(
    output_dir / "development_fit.npz",
    coefficients=contact_model.theta, covariance=contact_model.covariance,
    batter_ids=contact_model.player_ids["batter_id"],
    pitcher_ids=contact_model.player_ids["pitcher_id"],
    batter_sides=contact_model.player_sides["batter_id"],
    pitcher_sides=contact_model.player_sides["pitcher_id"],
    precision=contact_model.precision,
)

### Prior plausibility and simulation recovery

The prior predictive check below uses the selected precision scales and declared context prior. Because scales were selected using validation outcomes, this is a check of the working prior assumptions rather than a pre-data prior elicitation.

Recovery simulation holds the real development matchup/context structure fixed, samples Gaussian player effects at the selected scales, and generates Bernoulli contacts. It checks the neutral-pitcher IZ fastball/LHP target for all batters, including sparse players. Twenty replications are a modest diagnostic. Report coverage by replication and sample-size band; players within a replication share estimated context effects, so the thousands of player results are not independent trials. Passing this check does not establish coverage with misspecification or account for penalty selection.

In [ ]:
diagnostic_rng = np.random.default_rng(45)
prior_draws = diagnostic_rng.normal(size=(contact_model.n_parameters, 500)) / np.sqrt(contact_model.precision[:, None])
prior_context_frame = pd.DataFrame(CONTEXT_LEVELS, columns=CONTEXT)
prior_design = contact_model.design_matrix(prior_context_frame, neutral_batter=True, neutral_pitcher=True)
prior_probabilities = expit(prior_design @ prior_draws)
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(prior_probabilities.ravel(), bins=30, edgecolor="white")
ax.set(xlabel="Prior contact probability", ylabel="Draw/context combinations", title="Working prior predictive probabilities")
plt.show()
print("Prior probability quantiles:", np.quantile(prior_probabilities, [0.01, 0.5, 0.99]))

recovery = simulation_recovery(contact_model, development, SIMULATION_REPETITIONS, seed=46)
recovery["sample_band"] = pd.cut(recovery.swing_count, bins=[0, 50, 200, np.inf], labels=["<50", "50–199", "200+"], right=False)
recovery_summary = recovery.groupby("sample_band", observed=True).agg(
    estimates=("covered_95", "size"), coverage_95=("covered_95", "mean"),
    mean_error=("error", "mean"), mean_interval_width=("interval_width", "mean"),
)
display(recovery_summary)
if recovery.covered_95.mean() < 0.90:
    print("Approximate intervals under-cover in recovery simulation; interpret cautiously and revisit the approximation.")
replication_coverage = recovery.groupby("replication").covered_95.mean()
print("Coverage across replications:")
display(replication_coverage.describe())
recovery.to_csv(output_dir / "simulation_recovery.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].scatter(recovery.true_probability, recovery.estimated_probability, s=4, alpha=0.15)
axes[0].plot([0, 1], [0, 1], "k--")
axes[0].set(xlabel="Known simulated probability", ylabel="Recovered probability", title="Simulation recovery")
axes[1].hist(replication_coverage, bins=10, edgecolor="white")
axes[1].axvline(0.95, color="black", linestyle="--")
axes[1].set(xlabel="Coverage per replication", ylabel="Replications", title="Approximate 95% interval coverage")
fig.tight_layout()
plt.show()

### Protected test evaluation across the complete context grid

Everything above, including model settings, is frozen before computing these scores. Score actual batter–pitcher predictions, not neutral-opponent projections. Report proper scores, discrimination, sample sizes, and batter-balanced scores. Evaluate all 16 contexts plus marginal splits, training sample-size bands, and the three requested settings. Test findings are assessment; revising the model afterward would require a new evaluation design.

In [ ]:
test_predictions = {"Overall": np.full(len(test), development.contact.mean())}
test_predictions.update({name: model.predict(test) for name, model in final_models.items()})
test_scores = pd.DataFrame({name: performance(test, prediction) for name, prediction in test_predictions.items()}).T
display(test_scores)
test_scores.to_csv(output_dir / "test_scores.csv")
heldout_probability = test_predictions["Context + batter + pitcher"]
context_scores = subgroup_performance(test, heldout_probability, CONTEXT)
display(context_scores)
context_scores.to_csv(output_dir / "test_context_scores.csv")
for feature in CONTEXT:
    print(f"Marginal split: {feature}")
    display(subgroup_performance(test, heldout_probability, [feature]))

sample_test = test.assign(training_swings=test.batter_id.map(train.groupby("batter_id").size()))
sample_test["sample_band"] = pd.cut(sample_test.training_swings, bins=[0, 50, 200, np.inf], labels=["<50", "50–199", "200+"], right=False)
print("Batter training sample-size bands:")
display(subgroup_performance(sample_test, heldout_probability, ["sample_band"]))
pitcher_sample_test = test.assign(training_swings=test.pitcher_id.map(train.groupby("pitcher_id").size()))
pitcher_sample_test["sample_band"] = pd.cut(pitcher_sample_test.training_swings, bins=[0, 50, 200, np.inf], labels=["<50", "50–199", "200+"], right=False)
print("Pitcher training sample-size bands:")
display(subgroup_performance(pitcher_sample_test, heldout_probability, ["sample_band"]))

pdf_masks = {
    "IZ": test.is_in_zone.eq(1),
    "IZ vs LHP": test.is_in_zone.eq(1) & test.throw_side.eq("L"),
    "IZ FB vs LHP": test.is_in_zone.eq(1) & test.throw_side.eq("L") & test.is_fb.eq(1),
}
pdf_scores = pd.DataFrame({name: performance(test.loc[mask], heldout_probability[mask]) for name, mask in pdf_masks.items()}).T
display(pdf_scores)
test.assign(predicted_contact=heldout_probability).rename_axis("original_pitch_row").to_csv(output_dir / "test_predictions.csv")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, metric in zip(axes, ["log_loss", "brier"]):
    ax.barh(test_scores.index, test_scores[metric])
    ax.set(xlabel=metric, title=f"Test {metric} (lower is better)")
fig.tight_layout()
plt.show()

### Calibration: overall, all 16 contexts, and systematic residuals

Reliability curves compare mean predicted probability with observed contact in equal-frequency bins. Bin counts are shown; small groups use fewer bins (roughly at least 30 observations per bin). These are descriptive curves, not confidence bands. Scores and calibration assess different aspects of performance; ROC AUC cannot establish calibration. No post-hoc calibrator is fit to test data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name in ["Context", "Context + batter + pitcher"]:
    curve = reliability(test, test_predictions[name])
    axes[0].plot(curve.predicted, curve.observed, "o-", label=name)
axes[0].plot([0, 1], [0, 1], "k--")
axes[0].set(xlim=(0, 1), ylim=(0, 1), xlabel="Predicted contact probability", ylabel="Observed contact fraction", title="Overall test reliability")
axes[0].legend()
axes[1].hist(heldout_probability, bins=20, edgecolor="white")
axes[1].set(xlabel="Predicted contact probability", ylabel="Test swings", title="Distribution of predictions")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(4, 4, figsize=(14, 13), sharex=True, sharey=True)
for ax, context in zip(axes.ravel(), CONTEXT_LEVELS):
    mask = np.ones(len(test), dtype=bool)
    for key, value in zip(CONTEXT, context):
        mask &= test[key].eq(value).to_numpy()
    subset = test.loc[mask]
    curve = reliability(subset, heldout_probability[mask], max_bins=8)
    ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
    ax.plot(curve.predicted, curve.observed, "o-")
    for row in curve.itertuples():
        ax.annotate(str(row.count), (row.predicted, row.observed), fontsize=6, xytext=(2, 4), textcoords="offset points")
    bat, throw, zone, fb = context
    ax.set_title(f"Bat {bat} / Throw {throw} / {'IZ' if zone else 'OOZ'} / {'FB' if fb else 'non-FB'}\nn={len(subset):,}", fontsize=9)
    ax.set(xlim=(0, 1), ylim=(0, 1))
fig.supxlabel("Predicted contact probability (labels show bin counts)")
fig.supylabel("Observed contact fraction")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 5))
labels = [f"Bat {b} / Throw {p} / {'IZ' if z else 'OOZ'} / {'FB' if f else 'non-FB'}" for b, p, z, f in context_scores.index]
ax.barh(labels, context_scores.observed_minus_predicted)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(xlabel="Observed minus predicted contact fraction", title="Test residuals for all 16 contexts")
fig.tight_layout()
plt.show()

### Replicated held-out outcomes

Generate held-out contacts using joint coefficient draws **and Bernoulli outcome noise**. The resulting intervals describe replicated contact fractions over these particular test swings; they are different from intervals for a batter's latent probability. Compare all 16 contexts. This check is conditional on the frozen model and does not trigger test-driven refitting.

In [ ]:
predictive_check = predictive_context_check(contact_model, test, coefficient_draws[:, :400], seed=47)
display(predictive_check)
predictive_check.to_csv(output_dir / "test_predictive_context_check.csv")
fig, ax = plt.subplots(figsize=(7, 4))
x = predictive_check.replicated_mean.to_numpy()
y = predictive_check.observed.to_numpy()
ax.hlines(y, predictive_check.lower_95, predictive_check.upper_95, alpha=0.6)
ax.scatter(x, y)
ax.plot([0, 1], [0, 1], "k--")
ax.set(xlabel="Replicated contact fraction (95% predictive interval)", ylabel="Observed test contact fraction", title="Held-out predictive check by context")
fig.tight_layout()
plt.show()

### Shrinkage and uncertainty by sample size

The first comparison uses a raw **logit effect relative to the fitted context baseline**, which makes regression toward the mean more interpretable than comparing raw and standardized probabilities with different pitch mixes. The raw estimate still contains historical opponent composition; the fitted effect adjusts for pitcher quality. The second plot compares raw IZ FB/LHP rates with neutral-pitcher projections and should therefore be read as both shrinkage and context/opposition adjustment. The interval-width plot examines the fully conditioned PDF target; uncertainty also depends on opposition and context coverage, not just total sample size.

In [ ]:
batter_effects = contact_model.player_effects("batter_id")
context_logits = contact_model.theta[:16]
context_number = (development.bat_side.eq("R").astype(int) * 8 + development.throw_side.eq("R").astype(int) * 4 + development.is_in_zone * 2 + development.is_fb)
raw_players = development.assign(population_probability=expit(context_logits[context_number.to_numpy()])).groupby("batter_id").agg(contacts=("contact", "sum"), swings=("contact", "size"), context_mean=("population_probability", "mean"))
# Jeffreys-style 0.5 correction prevents infinite logits for raw 0/1 rates.
raw_probability = (raw_players.contacts + 0.5) / (raw_players.swings + 1)
raw_effect = np.log(raw_probability / (1 - raw_probability)) - np.log(raw_players.context_mean / (1 - raw_players.context_mean))
raw_effect -= raw_effect.groupby(batter_effects.side).transform("mean")

# Internal diagnostics only: the user-facing output below is a single-batter lookup.
projection_rows = []
for player_id in contact_model.player_ids["batter_id"]:
    projection = batter_projection(contact_model, int(player_id), reference_mix, coefficient_draws)
    last = projection.iloc[-1]
    projection_rows.append({"batter_id": player_id, **last.to_dict()})
projection_diagnostics = pd.DataFrame(projection_rows).set_index("batter_id")
projection_diagnostics["training_swings"] = development.groupby("batter_id").size()
raw_target = development.loc[development.is_in_zone.eq(1) & development.is_fb.eq(1) & development.throw_side.eq("L")].groupby("batter_id").contact.agg(["mean", "size"])
projection_diagnostics["raw_contact"] = raw_target["mean"]
projection_diagnostics["target_swings"] = raw_target["size"].fillna(0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
points = axes[0].scatter(raw_effect, batter_effects.effect, c=np.log1p(batter_effects.swing_count), cmap="viridis", s=16)
limit = max(abs(raw_effect).max(), abs(batter_effects.effect).max())
axes[0].plot([-limit, limit], [-limit, limit], "k--")
axes[0].set(xlabel="Raw context-relative logit effect", ylabel="Shrunken batter effect", title="Player adjustment and shrinkage")
fig.colorbar(points, ax=axes[0], label="log(1 + development swings)")
axes[1].scatter(projection_diagnostics.raw_contact, projection_diagnostics.contact_probability, c=np.log1p(projection_diagnostics.target_swings), cmap="viridis", s=16)
axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set(xlabel="Raw IZ FB/LHP contact fraction", ylabel="Neutral-pitcher probability", title="Raw vs adjusted target")
axes[2].scatter(projection_diagnostics.training_swings, projection_diagnostics.upper_95 - projection_diagnostics.lower_95, s=16)
axes[2].set(xscale="log", xlabel="Development swings", ylabel="95% probability interval width", title="Uncertainty by sample size")
axes[2].set_xticks([20, 50, 100, 200, 500], ["20", "50", "100", "200", "500"])
axes[2].minorticks_off()
fig.tight_layout()
plt.show()

### Lookup 1: PDF answers by batter ID or batting side

For a particular batter, set `batter_id` to a known ID and leave `bat_side = None` to infer their side. For a batting-side lookup, set **`batter_id = None` and `bat_side = "L"` or `"R"`**. If both are supplied, they must agree.

Side-only mode predicts for a **neutral-quality batter of that side**, using the mean logit batter effect among training players of that side. Its interval measures uncertainty in this mean effect; individual players can vary around it. Both modes use neutral pitcher quality and the batting-side-specific development reference mix. All probabilities condition on a swing. Reference weights and selected shrinkage are held fixed in the approximate intervals.

Observed counts describe development swings for the player, or pooled across players of the chosen batting side. Unknown IDs and invalid inputs produce an informative message. Change the inputs below and rerun the cell.

Input examples for the next cell:

| Request | `batter_id` | `bat_side` |
| --- | --- | --- |
| Neutral LHH | `None` | `"L"` |
| Neutral RHH | `None` | `"R"` |
| Batter 1 (default example), using their observed batting side | `1` | `None` |


In [ ]:
# Edit these two inputs. This example answers the PDF questions for batter 1.
batter_id = 1  # Known batter; set to None for a batting-side-only lookup.
bat_side = None  # Infer from batter ID; with batter_id=None, choose "L" or "R".

try:
    batter_answers = batter_projection(
        contact_model, batter_id, reference_mix, coefficient_draws, bat_side=bat_side
    )
except ValueError as error:
    print(error)
    print("Use a known batter ID, or set batter_id=None and bat_side='L' or 'R'.")
else:
    if batter_id is None:
        player_swings = development.loc[development.bat_side.eq(bat_side)]
        side = bat_side
        subject = f"Neutral {side}HH"
        count_description = "pooled development swings"
    else:
        player_swings = development.loc[development.batter_id.eq(batter_id)]
        side = player_swings.bat_side.iloc[0]
        subject = f"Batter {batter_id}"
        count_description = "development swings"
    masks = [
        player_swings.is_in_zone.eq(1),
        player_swings.is_in_zone.eq(1) & player_swings.throw_side.eq("L"),
        player_swings.is_in_zone.eq(1) & player_swings.throw_side.eq("L") & player_swings.is_fb.eq(1),
    ]
    batter_answers["observed_swings"] = [int(mask.sum()) for mask in masks]
    print(f"{subject} | bats {side} | {len(player_swings):,} {count_description}")
    display(batter_answers.style.format({"contact_probability": "{:.1%}", "lower_95": "{:.1%}", "upper_95": "{:.1%}"}))
    fig, ax = plt.subplots(figsize=(8, 3))
    positions = np.arange(len(batter_answers))
    ax.hlines(positions, batter_answers.lower_95, batter_answers.upper_95, linewidth=3)
    ax.scatter(batter_answers.contact_probability, positions)
    ax.set_yticks(positions, batter_answers.index)
    ax.invert_yaxis()
    ax.set(xlim=(0, 1), xlabel="Contact probability | swing (approximate 95% interval)", title=f"{subject}: standardized neutral-pitcher projections")
    fig.tight_layout()
    plt.show()

### Lookup 2: configurable contact probability

Choose `context_bat_side` and `context_throw_side` as `"L"` or `"R"`, `context_is_in_zone` as **1 = IZ / 0 = OOZ**, and `context_is_fb` as **1 = fastball / 0 = non-fastball**. All 16 combinations are supported.

The example uses a neutral LHH against neutral RHP on an out-of-zone fastball. To assess a particular batter, set `context_batter_id` to their ID and `context_bat_side = None` to infer their side. Both the batter and pitcher are neutral quality when using side-only mode; a specified batter retains their fitted ability. Pitcher quality remains neutral within the requested throwing side.

This fixes every context feature, so it does not average over the reference mixture. The result is contact probability conditional on a swing, with a joint-draw approximate 95% interval. Counts show development swings matching the requested player or batting side and exact pitch context.

In [ ]:
# Edit these five inputs to choose any supported context.
context_batter_id = None  # Optional known batter ID; None uses neutral batting quality.
context_bat_side = "L"  # "L" / "R"; use None to infer from a supplied batter ID.
context_throw_side = "R"  # "L" / "R"; neutral pitcher quality within this side.
context_is_in_zone = 0  # 1 = IZ, 0 = OOZ.
context_is_fb = 1  # 1 = FB, 0 = non-FB.

try:
    context_answers = context_projection(
        contact_model, coefficient_draws,
        batter_id=context_batter_id, bat_side=context_bat_side,
        throw_side=context_throw_side, is_in_zone=context_is_in_zone, is_fb=context_is_fb,
    )
except ValueError as error:
    print(error)
else:
    resolved_side = context_answers.bat_side.iloc[0]
    if context_batter_id is None:
        matching_swings = development.loc[development.bat_side.eq(resolved_side)]
        context_subject = f"Neutral {resolved_side}HH"
    else:
        matching_swings = development.loc[development.batter_id.eq(context_batter_id)]
        context_subject = f"Batter {context_batter_id}"
    matching_swings = matching_swings.loc[
        matching_swings.throw_side.eq(context_throw_side)
        & matching_swings.is_in_zone.eq(context_is_in_zone)
        & matching_swings.is_fb.eq(context_is_fb)
    ]
    context_answers["observed_swings"] = len(matching_swings)
    print(f"{context_subject} vs neutral {'LHP' if context_throw_side == 'L' else 'RHP'}")
    display(context_answers.style.format({"contact_probability": "{:.1%}", "lower_95": "{:.1%}", "upper_95": "{:.1%}"}))

### Opponent-quality and neutral-batter examples

Hold the hitter and pitch context fixed while changing the pitcher. The converse holds pitcher/context fixed and varies batters. These comparisons illustrate jointly adjusted player quality. They are predictive associations, not causal estimates; neutral-opponent projections are not expected to equal raw rates against historical opposition.

All three comparisons include approximate 95% intervals for contact probability from the existing joint coefficient draws. The plot shows those intervals for the chosen actual pitchers. These are conditional probability intervals under the selected model, not intervals for a realized contact outcome.

In [ ]:
example_batter_id = 1
example_batter_side = development.loc[development.batter_id.eq(example_batter_id), "bat_side"].iloc[0]
pitcher_effects = contact_model.player_effects("pitcher_id")
example_pitchers = pitcher_effects.loc[pitcher_effects.side.eq("L")].sort_values("effect")
chosen_pitcher_ids = example_pitchers.iloc[[0, len(example_pitchers) // 2, -1]].index
matchups = pd.DataFrame({"batter_id": example_batter_id, "bat_side": example_batter_side, "pitcher_id": chosen_pitcher_ids, "throw_side": "L", "is_in_zone": 1, "is_fb": 1})

quality_example = matchups[["pitcher_id"]].copy()
quality_example["pitcher_contact_effect"] = pitcher_effects.loc[chosen_pitcher_ids, "effect"].to_numpy()
quality_example["batter_contact_probability"] = contact_model.predict(matchups)
matchup_draws = expit(contact_model.design_matrix(matchups) @ coefficient_draws)
quality_example["lower_95"] = np.quantile(matchup_draws, 0.025, axis=1)
quality_example["upper_95"] = np.quantile(matchup_draws, 0.975, axis=1)
display(quality_example.style.format({"pitcher_contact_effect": "{:.3f}", "batter_contact_probability": "{:.1%}", "lower_95": "{:.1%}", "upper_95": "{:.1%}"}))

neutral_pitcher_matchup = matchups.iloc[:1]
neutral_pitcher_draws = expit(
    contact_model.design_matrix(neutral_pitcher_matchup, neutral_pitcher=True) @ coefficient_draws
)
neutral_pitcher_example = pd.DataFrame({
    "batter_contact_probability": contact_model.predict(neutral_pitcher_matchup, neutral_pitcher=True),
    "lower_95": np.quantile(neutral_pitcher_draws, 0.025, axis=1),
    "upper_95": np.quantile(neutral_pitcher_draws, 0.975, axis=1),
}, index=["Same batter vs neutral LHP"])
display(neutral_pitcher_example.style.format("{:.1%}"))

neutral_batter_rows = pd.DataFrame({"bat_side": ["L", "R"], "pitcher_id": chosen_pitcher_ids[1], "throw_side": "L", "is_in_zone": 1, "is_fb": 1})
neutral_batter_rows["neutral_batter_contact_probability"] = contact_model.predict(neutral_batter_rows, neutral_batter=True)
neutral_batter_draws = expit(
    contact_model.design_matrix(neutral_batter_rows, neutral_batter=True) @ coefficient_draws
)
neutral_batter_rows["lower_95"] = np.quantile(neutral_batter_draws, 0.025, axis=1)
neutral_batter_rows["upper_95"] = np.quantile(neutral_batter_draws, 0.975, axis=1)
display(neutral_batter_rows.style.format({"neutral_batter_contact_probability": "{:.1%}", "lower_95": "{:.1%}", "upper_95": "{:.1%}"}))

fig, ax = plt.subplots(figsize=(6, 3))
ax.vlines(quality_example.pitcher_contact_effect, quality_example.lower_95, quality_example.upper_95, alpha=0.7)
ax.scatter(quality_example.pitcher_contact_effect, quality_example.batter_contact_probability)
for row in quality_example.itertuples():
    ax.annotate(f"Pitcher {row.pitcher_id}", (row.pitcher_contact_effect, row.batter_contact_probability), xytext=(3, 4), textcoords="offset points")
ax.set(xlabel="Pitcher effect: larger means more contact allowed", ylabel="Contact probability (95% interval)", title=f"Batter {example_batter_id}, IZ fastball vs LHP")
fig.tight_layout()
plt.show()